# Mosaic integration: RNA + ADT

Mosaic integration combines batches that share only some modalities. Each method accepts one batch pattern. This tutorial is for an RNA batch, an RNA + ADT batch and an ADT batch. It runs Multigrate on `D38mini`, 6,000 cells of the benchmark dataset `D38`, and then on data in your own format.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_mosaic_rna_adt.ipynb).

## 1. Install

This cell installs `multibench-sc`. It keeps the numpy and pandas that are already installed, so Colab needs no restart.

<details>
<summary>Details</summary>

On Colab, choose a GPU runtime before you run the notebook: Runtime -> Change runtime type -> T4 GPU. On a CPU runtime, an environment that has a smaller CPU build gets that build, and training methods run slower.

</details>

In [ ]:
import numpy, pandas
%pip install -q multibench-sc numpy=={numpy.__version__} pandas=={pandas.__version__}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D38mini` (2 MB) once. Each method runs in its own environment. `mtb.env.install` downloads the environment for Multigrate, with no conda needed. The download is 4.4 GB.

<details>
<summary>Details</summary>

`state` is `PACKED` for an environment downloaded now and `have` for one that was already there. Without `dry_run=False`, `mtb.env.install` downloads nothing and returns the plan with its sizes.

Environments go to `mtb.config.DEFAULT.envs_dir`. To use another disk, set it before this cell.

From a terminal, `multibench env install --methods Multigrate --packed --run` does the same.

</details>

In [ ]:
import pandas as pd
import multibench as mtb

mtb.data.fetch("D38mini")

In [ ]:
METHODS = ["Multigrate"]
envs = mtb.env.install(METHODS, dry_run=False)
pd.DataFrame(envs)[["env", "methods", "state"]]

## 3. Run the method

`run_all` runs each method on `D38mini` and scores its output with the scIB metrics. `res.summary` has one row per method with its status, run time and scores. Higher is better for every metric.

<details>
<summary>Details</summary>

Each method writes an embedding: a table of numbers with one row per cell. `run_all` scores it against the cell-type labels of `D38mini`.

The clustering metrics `ARI`, `NMI`, `ASW`, `iASW`, `iF1` and `cLISI` measure how well the embedding separates the cell types. The batch metrics `ASW_batch`, `GC` and `iLISI` measure how well the batches mix. They appear only when the data has several batches.

`res.failures` says why a method failed. `params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_batch("out/D38mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run_all("D38mini", "mosaic", methods=METHODS,
                  out_dir="out/D38mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size shows the rank within a column, and bigger is better. The fill compares a value with the other rows in the same column.

<details>
<summary>Details</summary>

The lightest fill is the lowest value in this figure, not zero.

Metrics are grouped by family: blue for dimension reduction and clustering, green for batch correction. Each family starts with an `Overall` bar. Its length and colour both show the family score.

A column whose rows all hold the same value is drawn grey, and the note under the figure names it. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw counts, with one AnnData per batch and a cell type for each cell. `mtb.io.export_dataset` with `batch_index=` writes one batch per call. Number the batches as below: RNA, then RNA + ADT, then ADT. Multigrate needs several thousand cells for a mosaic run.

<details>
<summary>Details: export</summary>

`mtb.describe_layout("mosaic")` lists the batch patterns and the methods that accept each one.

The folder name, `MYMOSAIC`, is the dataset name for `run_all`, and `data_path` is the folder that holds it.

`mtb.scan("MYMOSAIC", "mosaic", data_path="mydata")` checks the folder and the environments without running anything. Its `reason` column says what is missing.

`overwrite=True` replaces the files of an earlier run of this cell. Without it, `export_dataset` raises `FileExistsError` rather than replace a file.

</details>

Here every cell of `D38mini` takes the place of your data:

In [ ]:
d = mtb.config.DEFAULT.data_path / "D38mini"
rna1 = mtb.io.read_canonical(d / "rna1.h5")
rna2 = mtb.io.read_canonical(d / "rna2.h5")
adt2 = mtb.io.read_canonical(d / "adt2.h5")
adt3 = mtb.io.read_canonical(d / "adt3.h5")
labels = [pd.read_csv(d / f"cty{b}.csv")["x"].values for b in (1, 2, 3)]

Write the folder, then run the same method on it:

In [ ]:
out = "mydata/MYMOSAIC"
# batch 1: RNA only
mtb.io.export_dataset(rna1, out, labels=labels[0],
                      batch_index=1, category="mosaic", overwrite=True)
# batch 2: RNA + ADT
mtb.io.export_dataset(rna2, out, adt=adt2, labels=labels[1],
                      batch_index=2, category="mosaic", overwrite=True)
# batch 3: ADT only
mtb.io.export_dataset(adt3, out, rna=None, adt="X", labels=labels[2],
                      batch_index=3, category="mosaic", overwrite=True)

In [ ]:
mine = mtb.run_all("MYMOSAIC", "mosaic", methods=METHODS, data_path="mydata",
                   out_dir="out/MYMOSAIC")
mine.summary

In [ ]:
mine.plot()

## 6. Every method of this task

1 method has a variant for mosaic RNA + ADT: Multigrate. Each runs on `D38mini`. To run them all, set `METHODS` to that list in section 2 and run the notebook again. That downloads 1 env, 4.4 GB to download on a CPU host, 4.4 GB on a GPU host.

[Every method on `D38mini`](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_adt_all/) shows that run.

## Troubleshooting

`res.failures` lists each method that failed or was skipped. Its `error` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| `env.install` refuses on macOS or Windows | methods run only on Linux: use Colab or a Linux machine |
| a method needs an NVIDIA GPU | choose a GPU runtime on Colab, or a machine with a GPU |
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a warning that values are not whole numbers | export raw counts, for example with `rna="layer:counts"` |
| `... matrix/data as cells x features` | the matrix is transposed: export it again with `mtb.io.export_dataset` |
| a method times out | raise `timeout=` in `run_all` |
| low `label_order_confidence` | several label files fit the cell count: check `label_order_candidates` in `res.results` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- `mtb.cite(METHODS)` returns the citations for the benchmark and the methods you ran.
- The other tutorials: [vertical RNA + ADT](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_adt/), [vertical RNA + ATAC](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_atac/), [vertical RNA + ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/vertical_rna_adt_atac/), [diagonal RNA + ATAC](https://dsichang.github.io/scMultiBench/tutorials/diagonal_rna_atac/), [mosaic RNA + ATAC](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_atac/), [mosaic RNA + ADT + ATAC](https://dsichang.github.io/scMultiBench/tutorials/mosaic_rna_adt_atac/), [cross RNA + ADT](https://dsichang.github.io/scMultiBench/tutorials/cross_rna_adt/).
- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.